<a href="https://colab.research.google.com/github/fourmansyah/Mapperatorinator/blob/main/colab/mapperatorinator_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Beatmap Generation with Mapperatorinator

This notebook is an interactive demo of an osu! beatmap generation model created by OliBomby. This model is capable of generating hit objects, hitsounds, timing, kiai times, and SVs for all 4 gamemodes. You can upload a beatmap to give to the model as additional context or remap parts of the beatmap.

### Instructions for running:

* Read and accept the rules regarding using this tool by clicking the checkbox.
* Make sure to use a GPU runtime, click:  __Runtime >> Change Runtime Type >> GPU__
* __Execute each cell in order__. Press ▶️ on the left of each cell to execute the cell.
* __Setup Environment__: run the first cell to clone the repository and install the required dependencies. You only need to run this cell once per session.
* __Upload Audio__: choose a .mp3 or .ogg file from your computer.
* __Upload Beatmap__: optionally choose a beatmap .osu file from your computer.  You can find these files in stable by using File > Open Song Folder, or in lazer by using File > Edit Externally.
* __Configure__: choose your generation parameters to control the style of the generated beatmap.
* Generate the beatmap using the __Generate Beatmap__ cell. (it may take a few minutes depending on the length of the song)


In [ ]:
#@title Setup Environment { display-mode: "form" }
#@markdown ### Use this tool responsibly. Always disclose the use of AI in your beatmaps. Accept the rules and run this cell.
i_accept_the_rules = True # @param {type:"boolean"}

assert i_accept_the_rules, "Read and accept the rules first!"

!git clone https://github.com/fourmansyah/Mapperatorinator.git
%cd Mapperatorinator

!pip install transformers==4.57.3
!pip install hydra-core nnaudio
!pip install slider git+https://github.com/llllllllll/slider.git
!pip install rosu-pp-py git+https://github.com/MaxOhn/rosu-pp-py.git
!pip install yt-dlp
from google.colab import files

import os
from hydra import compose, initialize_config_dir
from osuT5.osuT5.event import ContextType
from inference import main

output_path = "output"
input_audio = ""
input_beatmap = ""

In [ ]:
#@title Upload Audio { display-mode: "form" }
#@markdown Run this cell to upload audio. This is the song to generate a beatmap for. Please upload a .mp3 or .ogg file.

def upload_audio():
    data = list(files.upload().keys())
    if len(data) > 1:
        print('Multiple files uploaded; using only one.')
    file = data[0]
    if not file.endswith('.mp3') and not file.endswith('.ogg'):
        print('Invalid file format. Please upload a .mp3 or .ogg file.')
        return ""
    return data[0]

input_audio = upload_audio()

In [ ]:
#@title Download Audio via URL { display-mode: "form" }
#@markdown Masukkan URL YouTube atau direct link file audio (.mp3/.ogg).
audio_url = "" #@param {type:"string"}

import os
import subprocess

def download_audio_from_url(url):
    if url.strip() == "":
        print("❌ URL kosong! Harap masukkan link yang valid.")
        return ""

    print(f"Mendownload audio dari: {url}")
    output_base_name = "audio"

    # Menghapus file lama jika ada, agar tidak tertumpuk
    if os.path.exists(f"{output_base_name}.mp3"):
        os.remove(f"{output_base_name}.mp3")

    try:
        # Perintah untuk yt-dlp: Ekstrak audio saja dan ubah ke mp3
        command = [
            "yt-dlp",
            "-x", "--audio-format", "mp3",
            "--audio-quality", "0", # Kualitas audio terbaik
            "-o", f"{output_base_name}.%(ext)s",
            url
        ]

        # Menjalankan perintah download
        subprocess.run(command, check=True)

        final_file = f"{output_base_name}.mp3"
        if os.path.exists(final_file):
            print(f"✅ Sukses! File audio berhasil didownload dan disimpan sebagai: {final_file}")
            return final_file
        else:
            print("❌ Gagal menemukan file hasil download.")
            return ""

    except subprocess.CalledProcessError as e:
        print(f"❌ Terjadi kesalahan saat mendownload: {e}")
        return ""

# Menyimpan nama file ke dalam variabel yang akan dipakai oleh Generator AI
input_audio = download_audio_from_url(audio_url)

In [ ]:
#@title (Optional) Upload Beatmap { display-mode: "form" }
#@markdown This step is required if you want to use `in_context` or `add_to_beatmap` to provide additional info to the model.
#@markdown It will also fill in any missing metadata and unknown values in the configuration using info of the reference beatmap.
#@markdown Please upload a **.osu** file. You can find the .osu file in the song folder in stable or by using File > Edit Externally in lazer.
use_reference_beatmap = False # @param {type:"boolean"}

def upload_beatmap():
    data = list(files.upload().keys())
    if len(data) > 1:
        print('Multiple files uploaded; using only one.')
    file = data[0]
    if not file.endswith('.osu'):
        print('Invalid file format. Please upload a .osu file.\nIn stable you can find the .osu file in the song folder (File > Open Song Folder).\nIn lazer you can find the .osu file by using File > Edit Externally.')
        return ""
    return file

if use_reference_beatmap:
    input_beatmap = upload_beatmap()
else:
    input_beatmap = ""

In [ ]:
#@title Configure and Generate Beatmap { display-mode: "form" }

#@markdown #### You can input -1 (or leave blank for text) to leave the value unknown.
#@markdown ---
#@markdown This is the AI model to use. V30 is the most accurate model, but it does not support other gamemodes, year, descriptors, or in_context.
model = "Mapperatorinator V30" # @param ["Mapperatorinator V29", "Mapperatorinator V30", "Mapperatorinator V31"]
#@markdown This is the game mode to generate a beatmap for.
gamemode = "standard" # @param ["standard", "taiko", "catch the beat", "mania"]

#@markdown #### Metadata & Map Settings
#@markdown This is the title of the song. Leave blank to use auto-detection or reference map.
title = "" # @param {type:"string"}
#@markdown This is the artist of the song. Leave blank to use auto-detection or reference map.
artist = "" # @param {type:"string"}
#@markdown This is the creator (Mapper Name) of the beatmap.
creator = "AI" # @param {type:"string"}
#@markdown This is the version (Difficulty Name) of the beatmap, e.g., "Insane" or "Expert".
version_name = "AI Generated" # @param {type:"string"}
#@markdown Tags for searching and categorizing in osu!. Leave blank to ignore.
tags = "" # @param {type:"string"}
#@markdown Full path to the background image you want to use. Leave blank to ignore.
background = "" # @param {type:"string"}
#@markdown Time in milliseconds to start previewing the song in song select.
preview_time = -1 # @param {type:"integer"}

#@markdown #### Style & Difficulty
#@markdown This is the Star Rating you want your beatmap to be.
difficulty = 5 # @param {type:"number"}
#@markdown This is the user ID of the ranked mapper to imitate for mapping style.
mapper_id = -1 # @param {type:"integer"}
#@markdown This is the Beatmap ID to use as style (optional).
beatmap_id = -1 # @param {type:"integer"}
#@markdown This is the year you want the beatmap to be from (2007 - 2023).
year = 2023 # @param {type:"integer"}
#@markdown Optional HuggingFace repository ID of a LoRA to be applied to the current model.
lora_path = "" # @param {type:"string"}
#@markdown This is whether you want the beatmap to be hitsounded. This works only for mania mode.
hitsounded = True # @param {type:"boolean"}
#@markdown Use diffusion to generate object positions. True is recommended.
generate_positions = True # @param {type:"boolean"}

#@markdown #### Base Difficulty Settings (Editor)
hp_drain_rate = 5 # @param {type:"number"}
circle_size = 4 # @param {type:"number"}
overall_difficulty = 9 # @param {type:"number"}
approach_rate = 8 # @param {type:"number"}
slider_multiplier = 1.4 # @param {type:"slider", min:0.4, max:3.6, step:0.1}
slider_tick_rate = 1 # @param {type:"number"}

#@markdown #### Mania & Taiko Specifics
#@markdown This is the number of keys for the mania beatmap.
keycount = 4 # @param {type:"slider", min:1, max:18, step:1}
#@markdown This is the ratio of hold notes to circles in the beatmap [0,1].
hold_note_ratio = -1 # @param {type:"number"}
#@markdown This is the ratio of scroll speed changes to the number of notes [0,1].
scroll_speed_ratio = -1 # @param {type:"number"}

#@markdown #### Descriptors
descriptor_1 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
descriptor_2 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
descriptor_3 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
negative_descriptor_1 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
negative_descriptor_2 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}
negative_descriptor_3 = '' # @param ["slider only", "circle only", "collab", "megacollab", "marathon", "gungathon", "multi-song", "variable timing", "accelerating bpm", "time signatures", "storyboard", "storyboard gimmick", "keysounds", "download unavailable", "custom skin", "featured artist", "custom song", "style", "messy", "geometric", "grid snap", "hexgrid", "freeform", "symmetrical", "old-style revival", "clean", "slidershapes", "distance snapped", "iNiS-style", "avant-garde", "perfect stacks", "ninja spinners", "simple", "chaotic", "repetition", "progression", "high contrast", "improvisation", "playfield usage", "playfield constraint", "video gimmick", "difficulty spike", "low sv", "high sv", "colorhax", "tech", "slider tech", "complex sv", "reading", "visually dense", "overlap reading", "alt", "jump aim", "sharp aim", "wide aim", "linear aim", "aim control", "flow aim", "precision", "finger control", "complex snap divisors", "bursts", "streams", "spaced streams", "cutstreams", "stamina", "mapping contest", "tournament custom", "tag", "port"] {allow-input: true}

#@markdown #### Generation Bounds & Context
export_osz = False # @param {type:"boolean"}
add_to_beatmap = False # @param {type:"boolean"}
start_time = -1 # @param {type:"integer"}
end_time = -1 # @param {type:"integer"}
in_context = "[NONE]" # @param ["[NONE]", "[TIMING]", "[TIMING,KIAI]", "[TIMING,KIAI,MAP]", "[GD,TIMING,KIAI]", "[NO_HS,TIMING,KIAI]"]
output_type = "[MAP]" # @param ["[MAP]", "[TIMING,KIAI,MAP,SV]", "[TIMING]"]
cfg_scale = 1 # @param {type:"slider", min:1, max:5, step:0.1}
temperature = 1 # @param {type:"slider", min:0, max:1, step:0.01}
seed = -1 # @param {type:"integer"}

#@markdown #### Timing & BPM
timing_leniency = 20 # @param {type:"slider", min:0, max:100, step:1}
super_timing = False # @param {type:"boolean"}
timer_num_beams = 2 # @param {type:"slider", min:1, max:16, step:1}
timer_iterations = 20 # @param {type:"slider", min:1, max:100, step:1}
timer_bpm_threshold = 0.1 # @param {type:"slider", min:0, max:1, step:0.1}
bpm = -1 # @param {type:"number"}
#@markdown ---

# Get actual parameters
a_config = model.split(' ')[-1].lower()
a_gamemode = ["standard", "taiko", "catch the beat", "mania"].index(gamemode)
a_title = None if title == "" else title
a_artist = None if artist == "" else artist

# NEW: Tangkap parameter creator dan version
a_creator = None if creator == "" else creator
a_version = None if version_name == "" else version_name

a_tags = None if tags == "" else tags
a_background = None if background == "" else background
a_preview_time = None if preview_time == -1 else preview_time

a_difficulty = None if difficulty == -1 else difficulty
a_mapper_id = None if mapper_id == -1 else mapper_id
a_beatmap_id = None if beatmap_id == -1 else beatmap_id
a_year = None if year == -1 else year
a_lora_path = None if lora_path == "" else lora_path

a_hp_drain_rate = None if hp_drain_rate == -1 else hp_drain_rate
a_circle_size = None if circle_size == -1 else circle_size
a_overall_difficulty = None if overall_difficulty == -1 else overall_difficulty
a_approach_rate = None if approach_rate == -1 else approach_rate
a_slider_multiplier = None if slider_multiplier == -1 else slider_multiplier
a_slider_tick_rate = None if slider_tick_rate == -1 else slider_tick_rate
a_hold_note_ratio = None if hold_note_ratio == -1 else hold_note_ratio
a_scroll_speed_ratio = None if scroll_speed_ratio == -1 else scroll_speed_ratio

descriptors = [d for d in [descriptor_1, descriptor_2, descriptor_3] if d != '']
negative_descriptors = [d for d in [negative_descriptor_1, negative_descriptor_2, negative_descriptor_3] if d != '']

a_start_time = None if start_time == -1 else start_time
a_end_time = None if end_time == -1 else end_time
a_in_context = [ContextType(c.lower()) for c in in_context[1:-1].split(',')]
a_output_type = [ContextType(c.lower()) for c in output_type[1:-1].split(',')]
a_seed = None if seed == -1 else seed
a_bpm = None if bpm == -1 else bpm

# Validate stuff
if any(c in a_in_context for c in [ContextType.TIMING, ContextType.KIAI, ContextType.MAP, ContextType.SV, ContextType.GD, ContextType.NO_HS]) or add_to_beatmap:
    assert os.path.exists(input_beatmap), "Please upload a reference beatmap."
assert os.path.exists(input_audio), "Please upload an audio file."

if a_config == "v30":
    assert a_gamemode == 0, "V30 only supports standard mode."
    if any(c in a_in_context for c in [ContextType.KIAI, ContextType.MAP, ContextType.SV]):
        print("WARNING: V30 does not support KIAI, MAP, or SV in_context, ignoring.")
    if output_type != "[MAP]":
        print("WARNING: V30 only supports [MAP] output type, setting output type to [MAP].")
        a_output_type = [ContextType.MAP]
    if len(descriptors) != 0 and len(negative_descriptors) != 0:
        print("WARNING: V30 does not support descriptors or negative descriptors, ignoring.")
    if super_timing:
        print("WARNING: V30 does not fully support super timing, generation will be VERY slow.")

# Create config
with initialize_config_dir(version_base="1.1", config_dir="/content/Mapperatorinator/configs/inference"):
    conf = compose(config_name=a_config)

# Map all attributes to conf object
conf.audio_path = input_audio
conf.output_path = output_path
conf.beatmap_path = input_beatmap

# Apply Metadata & Map Settings
conf.gamemode = a_gamemode
conf.title = a_title
conf.artist = a_artist

# NEW: Masukkan creator dan version ke objek konfig
conf.creator = a_creator
conf.version = a_version

conf.tags = a_tags
conf.background = a_background
conf.preview_time = a_preview_time

# Apply Style & Difficulty
conf.difficulty = a_difficulty
conf.mapper_id = a_mapper_id
conf.beatmap_id = a_beatmap_id
conf.year = a_year
conf.lora_path = a_lora_path
conf.hitsounded = hitsounded
conf.generate_positions = generate_positions

# Apply Editor Stats
conf.hp_drain_rate = a_hp_drain_rate
conf.circle_size = a_circle_size
conf.overall_difficulty = a_overall_difficulty
conf.approach_rate = a_approach_rate
conf.slider_multiplier = a_slider_multiplier
conf.slider_tick_rate = a_slider_tick_rate
conf.keycount = keycount
conf.hold_note_ratio = a_hold_note_ratio
conf.scroll_speed_ratio = a_scroll_speed_ratio

# Apply Descriptors & Bounds
conf.descriptors = descriptors
conf.negative_descriptors = negative_descriptors
conf.export_osz = export_osz
conf.add_to_beatmap = add_to_beatmap
conf.start_time = a_start_time
conf.end_time = a_end_time
conf.in_context = a_in_context
conf.output_type = a_output_type
conf.cfg_scale = cfg_scale
conf.temperature = temperature
conf.seed = a_seed

# Apply Timing
conf.timing_leniency = timing_leniency
conf.super_timing = super_timing
conf.timer_num_beams = timer_num_beams
conf.timer_iterations = timer_iterations
conf.timer_bpm_threshold = timer_bpm_threshold
conf.bpm = a_bpm

# Run Inference
_, result_path, osz_path = main(conf)

if osz_path is not None:
    result_path = osz_path

if conf.add_to_beatmap:
    files.download(result_path)
else:
    files.download(result_path)